<a href="https://colab.research.google.com/github/JulianNguyen05/xiangqi-sorting-jetmax/blob/main/training/rules_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BƯỚC 5 — LUẬT CỜ TƯỚNG, KIỂM TRA THẾ CỜ VÀ ENGINE RA NƯỚC ĐI

**Đầu vào:** thế cờ dạng FEN (do bước nhận diện ảnh → FEN cung cấp) và thế cờ ở lượt trước
**Đầu ra:** nước đi người vừa thực hiện (đã kiểm tra hợp lệ) và **nước đi tiếp theo** cho robot
**Mã nguồn dùng trong notebook** (chạy được cả trên robot, Python 3.6, không cần thư viện ngoài):

| File | Việc |
|---|---|
| `board/rules.py` | Sinh nước đi hợp lệ, perft, kiểm tra thế cờ, suy ra nước vừa đi, ký hiệu tiếng Việt |
| `brain/uci.py` | Điều khiển engine qua giao thức UCI (Pikafish) |
| `brain/simple.py` | Engine dự phòng bằng Python (alpha-beta), dùng khi không chạy được Pikafish |
| `brain/game.py` | `GameTracker`: theo dõi ván cờ người – robot |
| `tools/build_pikafish.sh` | Build Pikafish cho Jetson Nano trên máy có mạng (link tĩnh, chép sang robot) |
| `tools/bench_engine.py` | Đo tốc độ engine trên robot |

### Nội dung notebook
1. Vị trí của bước này trong hệ thống
2. Biểu diễn thế cờ: toạ độ, FEN, ánh xạ 14 lớp YOLO → ký tự FEN
3. Luật đi của từng quân và **kiểm chứng bộ sinh nước đi bằng perft**
4. Kiểm tra một thế cờ nhận diện được có thể xảy ra trong ván thật không
5. Suy ra nước vừa đi từ 2 lần nhìn bàn cờ và **tự sửa lỗi nhận diện** (+ thực nghiệm mô phỏng)
6. Engine Pikafish: build, tốc độ theo độ sâu, đối chiếu luật, ván tự đánh, vòng lặp người – robot
7. Đưa engine lên Jetson Nano (robot không có mạng)

---
# 1. Vị trí trong hệ thống

Bài toán của đề tài: **ảnh bàn cờ → nước đi tiếp theo**. Có thể tách thành 4 tầng; notebook này làm tầng 3 và 4, chỉ làm việc với **thế cờ** (không cần ảnh, không cần GPU):

| Tầng | Việc | Ở đâu |
|---|---|---|
| 1 | Định vị bàn cờ, tính 90 giao điểm | `vision/board_detect.py` |
| 2 | YOLO11n nhận diện quân → gán vào giao điểm → FEN | `vision/pieces.py`, `vision/world.py` (Bước 2, 4) |
| **3** | **Kiểm tra hợp lệ, suy ra nước người vừa đi, sửa lỗi nhận diện** | **`board/rules.py`** |
| **4** | **Tìm nước đi tiếp theo** | **`brain/uci.py` + Pikafish** |

Lý do cần tầng 3: mô hình ở Bước 2 có Recall 0.961, tức cứ khoảng 25 quân lại bỏ sót 1 quân. Một bàn có 32 quân nên **phần lớn ảnh sẽ có ít nhất 1 ô sai** (mục 5.3). Nếu đưa thẳng FEN sai vào engine, robot sẽ đi theo một thế cờ không có thật. Tầng 3 dùng luật cờ để phát hiện và sửa các lỗi này.

In [ ]:
# ── Sơ đồ luồng xử lý (chỉ để minh hoạ)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(13, 2.8)); ax.axis('off'); ax.set_xlim(0, 13); ax.set_ylim(0, 2.8)
boxes = [(0.1, 'Ảnh camera', '#e8e8e8'), (2.0, 'Lưới 9×10\n+ YOLO11n', '#e8e8e8'), (3.9, 'FEN\nnhận diện', '#e8e8e8'),
         (5.8, 'Kiểm tra +\nsuy ra nước vừa đi', '#ffe0b2'), (8.0, 'Pikafish\n(UCI)', '#ffe0b2'), (10.0, 'Nước đi\n→ tay máy', '#e8e8e8')]
for x, t, c in boxes:
    w = 2.0 if 'Kiểm' in t else 1.6
    ax.add_patch(FancyBboxPatch((x, 0.9), w, 1.1, boxstyle='round,pad=0.05', fc=c, ec='#555'))
    ax.text(x + w / 2, 1.45, t, ha='center', va='center', fontsize=10)
for x0, x1 in [(1.7, 2.0), (3.6, 3.9), (5.5, 5.8), (7.8, 8.0), (9.6, 10.0)]:
    ax.annotate('', xy=(x1, 1.45), xytext=(x0, 1.45), arrowprops=dict(arrowstyle='->', lw=1.5))
ax.annotate('', xy=(6.8, 0.85), xytext=(9.0, 0.85), arrowprops=dict(arrowstyle='->', lw=1, ls='--', color='#c62828'))
ax.text(7.9, 0.45, 'thế cờ đã xác nhận ở lượt trước', ha='center', fontsize=9, color='#c62828')
ax.text(6.8, 2.35, 'notebook này (tô cam)', ha='center', fontsize=10, color='#e65100')
plt.show()

## Chuẩn bị môi trường

Trên Colab, ô dưới tải repo về `/content/xiangqi-sorting-jetmax`. Khi chạy trên laptop, mở notebook từ thư mục `training/` của repo.

In [ ]:
import os, sys, time, random, subprocess
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    REPO = '/content/xiangqi-sorting-jetmax'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1',
                        'https://github.com/JulianNguyen05/xiangqi-sorting-jetmax', REPO], check=True)
else:
    REPO = os.path.abspath('..') if os.path.exists('../board/rules.py') else os.path.abspath('.')
assert os.path.exists(f'{REPO}/board/rules.py'), 'Repo chưa có board/rules.py — hãy commit các file mới trước'
sys.path.insert(0, REPO)
os.chdir(REPO)

from board import rules as R
from board.geometry import FILES, NF, NR, square_name
from brain.uci import UciEngine
from brain.simple import SimpleEngine
from brain.game import GameTracker
print('Repo:', REPO)

---
# 2. Biểu diễn thế cờ

### 2.1 Toạ độ và FEN

| Quy ước | Giá trị | Ghi chú |
|---|---|---|
| Ô | `(f, r)`: cột `f` = 0..8 (a..i), hàng `r` = 0..9 | Giống `board/geometry.py`; **Đỏ ở hàng 0..4, phía robot** |
| Tên ô | `e0`, `h2`, … | Cột + hàng — đúng ký hiệu Pikafish dùng |
| Nước đi | `h2e2` = từ h2 đến e2 | Định dạng UCI |
| Mã quân | `K A B N R C P` = Tướng Sĩ Tượng Mã Xe Pháo Tốt | Chữ HOA = Đỏ, thường = Đen |
| FEN | `rnbakabnr/9/1c5c1/p1p1p1p1p/9/9/P1P1P1P1P/1C5C1/9/RNBAKABNR w - - 0 1` | Liệt kê từ hàng 9 xuống hàng 0; số = số ô trống; `w` = Đỏ đi, `b` = Đen đi |

### 2.2 Ánh xạ 14 lớp của YOLO11n (Bước 2) sang ký tự FEN

| id | lớp | FEN | id | lớp | FEN |
|---|---|---|---|---|---|
| 0 | black_advisor | a | 7 | red_advisor | A |
| 1 | black_bishop | b | 8 | red_bishop | B |
| 2 | black_cannon | c | 9 | red_cannon | C |
| 3 | black_king | k | 10 | red_king | K |
| 4 | black_knight | n | 11 | red_knight | N |
| 5 | black_pawn | p | 12 | red_pawn | P |
| 6 | black_rook | r | 13 | red_rook | R |

⚠ Thứ tự này phải trùng với `CLASSES` trong `build_dataset.ipynb` và `vision/pieces.py` (ô dưới kiểm tra lại).

In [ ]:
from vision.pieces import CLASSES, CODES
print(pd.DataFrame({'lớp': CLASSES, 'FEN': CODES}).T.to_string(header=False))

board, side = R.parse_fen(R.START)
assert R.to_fen(board, side) == R.START          # đọc FEN rồi ghi lại phải ra đúng chuỗi cũ
print('\nThế khai cuộc:', len(board), 'quân, bên đi:', 'Đỏ' if side == R.RED else 'Đen')

Hàm vẽ bàn cờ dùng trong cả notebook: quân Đỏ / Đen là các vòng tròn có ký tự FEN, chấm xanh là ô có thể đi tới, dấu × đỏ là ô ăn quân.

In [ ]:
def draw_board(ax, board, title='', moves=None, arrows=None, marks=None, mark_color='#d81b60'):
    ax.set_xlim(-0.8, 8.8); ax.set_ylim(-0.8, 9.8); ax.set_aspect('equal'); ax.axis('off')
    ax.add_patch(plt.Rectangle((-0.6, -0.6), 9.2, 10.2, fc='#f3d9a4', ec='none', zorder=0))
    for r in range(10):
        ax.plot([0, 8], [r, r], c='#5d4037', lw=0.8, zorder=1)
    for f in range(9):
        if f in (0, 8):
            ax.plot([f, f], [0, 9], c='#5d4037', lw=0.8, zorder=1)
        else:
            ax.plot([f, f], [0, 4], c='#5d4037', lw=0.8, zorder=1); ax.plot([f, f], [5, 9], c='#5d4037', lw=0.8, zorder=1)
    for r0 in (0, 7):                                              # đường chéo trong cung
        ax.plot([3, 5], [r0, r0 + 2], c='#5d4037', lw=0.8); ax.plot([5, 3], [r0, r0 + 2], c='#5d4037', lw=0.8)
    ax.text(4, 4.5, 'S Ô N G', ha='center', va='center', fontsize=8, color='#8d6e63', zorder=1)
    for f in range(9):
        ax.text(f, -0.75, FILES[f], ha='center', va='center', fontsize=7, color='#555')
    for r in range(10):
        ax.text(-0.75, r, str(r), ha='center', va='center', fontsize=7, color='#555')
    for (f, r), c in board.items():
        red = c.isupper()
        ax.add_patch(plt.Circle((f, r), 0.42, fc='#fff8e1', ec='#c62828' if red else '#212121', lw=1.6, zorder=3))
        ax.text(f, r, c, ha='center', va='center', fontsize=9, fontweight='bold',
                color='#c62828' if red else '#212121', zorder=4)
    for mv in moves or []:
        (f1, r1) = mv[1]
        if mv[1] in board:
            ax.plot(f1, r1, marker='x', ms=14, mew=2.5, c='#e53935', zorder=5)
        else:
            ax.plot(f1, r1, 'o', ms=7, c='#2e7d32', zorder=5)
    for (a, b) in arrows or []:
        ax.annotate('', xy=b, xytext=a, zorder=6, arrowprops=dict(arrowstyle='->', lw=2.2, color='#1565c0'))
    for (f, r) in marks or []:
        ax.add_patch(plt.Circle((f, r), 0.5, fc='none', ec=mark_color, lw=2.5, ls='--', zorder=6))
    ax.set_title(title, fontsize=10)

fig, ax = plt.subplots(figsize=(4, 4.6))
draw_board(ax, board, 'Thế khai cuộc (Đỏ ở hàng 0, phía robot)')
plt.show()

---
# 3. Bộ sinh nước đi hợp lệ

## 3.1 Luật đi của từng quân

| Quân | Cách đi | Điều kiện bị cản / giới hạn (dễ cài sai nhất) |
|---|---|---|
| Tướng (K) | 1 ô ngang/dọc | Chỉ trong cung 3×3; **hai tướng không được đối mặt** trên cùng cột mà không có quân chắn |
| Sĩ (A) | 1 ô chéo | Chỉ trong cung (5 điểm) |
| Tượng (B) | 2 ô chéo | Không qua sông; **bị cản nếu ô giữa ("mắt tượng") có quân** |
| Mã (N) | 1 thẳng + 1 chéo (hình chữ L) | **Bị cản nếu ô liền kề theo hướng đi thẳng ("chân mã") có quân** |
| Xe (R) | Thẳng, bao xa cũng được | Dừng khi gặp quân |
| Pháo (C) | Đi như xe khi không ăn | **Ăn quân phải nhảy qua đúng 1 quân ("ngòi")** |
| Tốt (P) | Tiến 1 ô | Qua sông mới được đi ngang; không bao giờ lùi |

**Nước đi hợp lệ** = đi đúng cách của quân **và** sau khi đi, tướng của mình không bị chiếu (kể cả lỗi để hai tướng đối mặt). Cách cài trong `board/rules.py`: sinh nước theo cách đi (`pseudo_moves`) → đi thử → kiểm tra `is_attacked(ô tướng)` bằng cách nhìn ngược từ ô tướng ra 4 hướng thẳng (xe, pháo, tướng), 8 vị trí mã (có xét chân mã) và các ô tốt. Cách nhìn ngược này nhanh hơn nhiều so với sinh toàn bộ nước của đối phương.

Ô dưới minh hoạ từng quân trong một thế cờ nhỏ có quân cản (chấm xanh = đi được, × = ăn được, vòng hồng = quân cản):

In [ ]:
def demo(fen_body, sq, title, blockers=()):
    b, _ = R.parse_fen(fen_body + ' w - - 0 1')
    mv = [m for m in R.legal_moves(b, R.RED) if m[0] == sq]
    return b, mv, title, blockers

P = lambda s: R.parse_square(s)
cases = [
    demo('5k3/9/9/9/4P4/9/1n2R4/9/9/3K5', P('e3'), 'Xe e3: dừng trước quân mình (e5),\nngang ăn mã b3', [P('e5')]),
    demo('5k3/9/9/9/9/4P4/4N4/6r2/9/3K5', P('e3'), 'Mã e3: tốt e4 chặn chân → mất 2 nước\nlên trên; ăn được xe g2', [P('e4')]),
    demo('5k3/9/1r7/9/9/1P7/9/9/1C2A2n1/3K5', P('b1'), 'Pháo b1: nhảy ngòi b4 ăn xe b7,\nnhảy ngòi e1 ăn mã h1', [P('b4'), P('e1')]),
    demo('5k3/9/9/9/9/2B6/3N5/9/9/3K5', P('c4'), 'Tượng c4: mắt d3 bị cản,\nkhông qua sông (a6, e6)', [P('d3')]),
    demo('5k3/9/9/9/9/9/9/9/4A4/3K5', P('e1'), 'Sĩ e1: 4 góc cung,\ntrừ d0 có tướng'),
    demo('5k3/9/9/9/9/9/9/9/4K4/9', P('e1'), 'Tướng e1: không sang f1 vì\nđối mặt tướng Đen f9'),
    demo('5k3/9/9/2P6/9/9/4P4/9/9/3K5', P('c6'), 'Tốt c6 (đã qua sông):\ntiến + đi ngang'),
    demo('5k3/9/9/2P6/9/9/4P4/9/9/3K5', P('e3'), 'Tốt e3 (chưa qua sông):\nchỉ tiến'),
]
fig, axes = plt.subplots(2, 4, figsize=(16, 9.5))
for ax, (b, mv, title, bl) in zip(axes.ravel(), cases):
    draw_board(ax, b, title, moves=mv, marks=bl)
plt.tight_layout(); plt.show()

## 3.2 Kiểm chứng bằng perft

**Perft** (*performance test*) đếm số chuỗi nước đi hợp lệ có thể có tới độ sâu $d$. Đây là phép kiểm tra chuẩn của mọi chương trình cờ: chỉ cần **một** luật cài sai (ví dụ quên chân mã) thì con số ở độ sâu 3–4 sẽ lệch. Giá trị chuẩn cho thế khai cuộc cờ tướng (Chessprogramming wiki, *Chinese Chess Perft Results*):

| Độ sâu | Số nút chuẩn |
|---|---|
| 1 | 44 |
| 2 | 1 920 |
| 3 | 79 666 |
| 4 | 3 290 240 |

Độ sâu 4 mất khoảng 30 giây bằng Python thuần; đặt `RUN_PERFT4 = False` nếu muốn bỏ qua.

In [ ]:
RUN_PERFT4 = True
EXPECTED = {1: 44, 2: 1920, 3: 79666, 4: 3290240}
rows = []
b0, s0 = R.parse_fen(R.START)
for d, exp in EXPECTED.items():
    if d == 4 and not RUN_PERFT4:
        continue
    t0 = time.perf_counter(); n = R.perft(b0, s0, d); dt = time.perf_counter() - t0
    rows.append({'độ sâu': d, 'board/rules.py': n, 'chuẩn': exp, 'khớp': '✅' if n == exp else '❌', 'thời gian (s)': round(dt, 2)})
perft_df = pd.DataFrame(rows); print(perft_df.to_string(index=False))
assert all(perft_df['board/rules.py'] == perft_df['chuẩn'])

Perft ở thế khai cuộc chưa đủ: ở đầu ván không có chiếu tướng, không có tốt qua sông, ít quân bị cản. Mục 6.4 đối chiếu thêm với **Pikafish** trên hàng nghìn thế cờ giữa trận sinh ngẫu nhiên.

---
# 4. Kiểm tra thế cờ nhận diện được

Ảnh → FEN có thể sai theo nhiều kiểu: thiếu quân (bỏ sót), thừa quân (báo nhầm), sai loại (ví dụ `black_cannon` – lớp yếu nhất ở Bước 2). `R.validate()` kiểm tra những điều **chắc chắn không thể xảy ra** trong ván thật:

| Mã lỗi | Điều kiện | Kiểu lỗi nhận diện thường gặp |
|---|---|---|
| `too_many` | Quá số quân của bộ cờ (2 xe, 2 pháo, 5 tốt…) | Nhầm loại (tốt → pháo) |
| `no_king` | Thiếu tướng | Bỏ sót tướng |
| `bad_square` | Tướng/sĩ ngoài các điểm trong cung; tượng ngoài 7 điểm hoặc qua sông; tốt lùi sau hàng xuất phát, hoặc chưa qua sông mà ở cột b d f h | Nhầm loại hoặc nhầm màu |
| `kings_facing` | Hai tướng đối mặt | Bỏ sót quân chắn giữa |
| `side_in_check` | Bên **vừa đi** đang bị chiếu (luật không cho phép) | Nhầm loại, nhầm màu |

⚠ Thế cờ hợp lệ **chưa chắc đúng**: nhầm mã thành xe ở cùng ô vẫn hợp lệ. Để bắt được loại lỗi này cần biết thế cờ trước đó (mục 5).

In [ ]:
START_BODY = R.START.split()[0]
tests = [
    ('Thế khai cuộc (đúng)',                       R.START),
    ('Bỏ sót tướng Đỏ e0',                          'rnbakabnr/9/1c5c1/p1p1p1p1p/9/9/P1P1P1P1P/1C5C1/9/RNBA1ABNR w - - 0 1'),
    ('Tốt Đen c6 bị nhận nhầm thành pháo',          'rnbakabnr/9/1c5c1/p1c1p1p1p/9/9/P1P1P1P1P/1C5C1/9/RNBAKABNR w - - 0 1'),
    ('Tượng Đỏ ở c5 (qua sông)',                    'rnbakabnr/9/1c5c1/p1p1p1p1p/2B6/9/P1P1P1P1P/1C5C1/9/RN1AKABNR w - - 0 1'),
    ('Tốt Đỏ ở b3 (chưa qua sông, cột chẵn)',       'rnbakabnr/9/1c5c1/p1p1p1p1p/9/9/PP2P1P1P/1C5C1/9/RNBAKABNR w - - 0 1'),
    ('Sót tốt e3 và e6 → hai tướng đối mặt',        'rnbakabnr/9/1c5c1/p1p3p1p/9/9/P1P3P1P/1C5C1/9/RNBAKABNR w - - 0 1'),
    ('Đỏ đi nhưng Đen đang bị chiếu',               '4k4/9/9/9/9/9/9/9/4R4/3K5 w - - 0 1'),
]
rows = []
for name, fen in tests:
    b, s = R.parse_fen(fen)
    iss = R.validate(b, s)
    rows.append({'trường hợp': name, 'hợp lệ': '✅' if not iss else '❌',
                 'lỗi phát hiện': '; '.join(i[2] for i in iss) or '—'})
pd.set_option('display.max_colwidth', 120)
display(pd.DataFrame(rows))

---
# 5. Suy ra nước vừa đi và tự sửa lỗi nhận diện

## 5.1 Ý tưởng

Robot luôn biết **thế cờ ở lượt trước** (đã xác nhận). Từ thế đó, bên đến lượt chỉ có khoảng 40 nước hợp lệ. Thay vì tin thẳng vào ảnh mới, ta chọn nước đi mà thế cờ sau nước đó **giống ảnh mới nhất**:

$$m^* = \arg\min_{m \in \text{Hợp lệ}(S_{t-1}) \cup \{\varnothing\}} \; d\big(S_{t-1} \oplus m,\; \hat S_t\big)$$

- $S_{t-1}$: thế cờ đã xác nhận; $\hat S_t$: thế cờ vừa nhận diện (có thể sai); $\varnothing$: "chưa đi".
- $d$: số ô khác nhau (khoảng cách Hamming trên 90 ô).
- Nếu $d(m^*) = 0$: nhận diện đúng hoàn toàn. Nếu $0 < d(m^*) \le$ `max_cost`: **đã sửa** các ô sai. Nếu lớn hơn, hoặc có 2 nước cùng khoảng cách: **không chắc chắn** → chụp lại.

**Vì sao sửa được?** Một nước đi chỉ thay đổi 2 ô (ô đi và ô đến). Hai nước hợp lệ khác nhau cho ra 2 thế cờ khác nhau ít nhất 2 ô, nên sai 1 ô vẫn luôn chọn đúng nước. Sai 2 ô thì thường vẫn đúng (mục 5.3 đo bằng mô phỏng).

In [ ]:
prev, _ = R.parse_fen(R.START)
true_mv = R.uci_to_move('h2e2')                      # người chơi Đỏ đi "Pháo 2 bình 5"
truth = R.make_move(prev, true_mv)
observed = dict(truth)
observed[(1, 7)] = 'r'                               # lỗi 1: pháo Đen b7 bị nhận nhầm thành xe
del observed[(8, 6)]                                 # lỗi 2: bỏ sót tốt Đen i6
res = R.infer_move(prev, R.RED, observed)
print('Nước suy ra   :', res['uci'], '=', R.move_to_vn(prev, res['move']))
print('Số ô đã sửa   :', res['cost'], '| không chắc chắn:', res['ambiguous'])
print('5 ứng viên tốt nhất (số ô lệch, nước):', res['candidates'])
print('Thế cờ sau khi sửa khớp sự thật:', res['board'] == truth)

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
draw_board(axes[0], prev, 'Thế trước (đã xác nhận)')
draw_board(axes[1], observed, 'Ảnh mới nhận diện được\n(vòng hồng = ô sai)', marks=[(1, 7), (8, 6)])
draw_board(axes[2], res['board'], 'Sau khi suy ra nước %s\n(đã sửa %d ô)' % (res['uci'], res['cost']),
           arrows=[res['move']])
plt.tight_layout(); plt.show()

## 5.2 Ký hiệu tiếng Việt cho nước đi

Để hiển thị cho người chơi và ghi biên bản ván cờ, `R.move_to_vn()` đổi nước UCI sang ký hiệu truyền thống: cột đánh số **1–9 từ phải sang trái theo góc nhìn của bên đi**; *tiến* / *thoái* / *bình* (đi ngang). Quân đi thẳng (xe, pháo, tốt, tướng) ghi số bước; quân đi chéo (mã, tượng, sĩ) ghi cột đến; 2 quân cùng loại trên cùng cột ghi *trước* / *sau*.

In [ ]:
b, _ = R.parse_fen(R.START)
for u in ['h2e2', 'h0g2', 'a0a1', 'g3g4', 'c0e2', 'f0e1']:
    print(f'{u}  →  {R.move_to_vn(b, R.uci_to_move(u))}')
b2, _ = R.parse_fen('4k4/9/9/9/9/9/4R4/9/4R4/3K5 w - - 0 1')
print('e3e5 (2 xe cùng cột) →', R.move_to_vn(b2, R.uci_to_move('e3e5')))

## 5.3 Thực nghiệm mô phỏng: sửa lỗi tốt tới mức nào?

Chưa có bộ ảnh thật, nên ta **mô phỏng** lỗi của bước nhận diện trên các thế cờ sinh từ ván ngẫu nhiên:

| Kiểu lỗi | Xác suất | Ứng với chỉ số ở Bước 2 |
|---|---|---|
| Bỏ sót 1 quân | $e$ cho mỗi quân | $1 - \text{Recall}$ (Recall 0.961 → $e \approx 0.04$) |
| Nhận nhầm loại (giữ màu 70%, đổi màu 30%) | $e/3$ cho mỗi quân | $1 - \text{Precision}$ ≈ 0.014 |
| Báo nhầm 1 quân ở ô trống | $e$ cho mỗi ảnh | |

Với mỗi mức lỗi $e$, so sánh 2 cách:
- **Tin thẳng ảnh**: dùng $\hat S_t$ làm thế cờ → đúng khi ảnh không có ô sai nào.
- **Suy ra bằng luật** (`infer_move`, `max_cost = 2`): kết quả là *đúng nước*, *không chắc → chụp lại*, hoặc **sai mà không biết** (nguy hiểm nhất: robot sẽ đánh theo thế cờ sai).

In [ ]:
TYPES = 'karbncp'

def random_positions(n, seed=0):
    """Các cặp (thế trước, bên đi, nước đi) từ ván ngẫu nhiên (ưu tiên ăn quân để có thế cờ thưa)."""
    rnd = random.Random(seed); out = []
    while len(out) < n:
        b, s = R.start_board(), R.RED
        for ply in range(rnd.randint(2, 100)):
            mv = R.legal_moves(b, s)
            if not mv:
                break
            caps = [m for m in mv if m[1] in b]
            m = rnd.choice(caps) if caps and rnd.random() < 0.5 else rnd.choice(mv)
            if rnd.random() < 0.08:
                out.append((dict(b), s, m))
            b, s = R.make_move(b, m), R.other(s)
    return out[:n]

def corrupt(board, e, rnd):
    obs = {}
    for sq, c in board.items():
        u = rnd.random()
        if u < e:
            continue                                          # bỏ sót
        if u < e + e / 3:                                     # nhầm loại
            t = rnd.choice([x for x in TYPES if x != c.lower()])
            red = c.isupper() if rnd.random() < 0.7 else not c.isupper()
            c = t.upper() if red else t
        obs[sq] = c
    if rnd.random() < e:                                      # báo nhầm 1 quân ở ô trống
        empty = [(f, r) for f in range(NF) for r in range(NR) if (f, r) not in obs]
        t = rnd.choice(TYPES); obs[rnd.choice(empty)] = t.upper() if rnd.random() < 0.5 else t
    return obs

positions = random_positions(600, seed=0)
print('Số thế cờ mô phỏng:', len(positions), '| số quân trung bình:', np.mean([len(b) for b, _, _ in positions]).round(1))

rows = []
rnd = random.Random(1)
for e in [0.0, 0.01, 0.02, 0.04, 0.06, 0.08, 0.12]:
    exact = ok = unsure = wrong = 0
    for b, s, mv in positions:
        truth = R.make_move(b, mv)
        obs = corrupt(truth, e, rnd)
        exact += obs == truth
        r = R.infer_move(b, s, obs)
        if r['cost'] > 2 or r['ambiguous']:
            unsure += 1
        elif r['move'] == mv:
            ok += 1
        else:
            wrong += 1
    n = len(positions)
    rows.append({'e': e, 'Tin thẳng ảnh: đúng': exact / n, 'Suy ra: đúng nước': ok / n,
                 'Suy ra: không chắc → chụp lại': unsure / n, 'Suy ra: SAI mà không biết': wrong / n})
sim = pd.DataFrame(rows)
display(sim.style.format({c: '{:.1%}' for c in sim.columns if c != 'e'}))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = sim['e'] * 100
ax.plot(x, sim['Tin thẳng ảnh: đúng'] * 100, 'o--', c='#757575', label='Tin thẳng ảnh: thế cờ đúng')
ax.plot(x, sim['Suy ra: đúng nước'] * 100, 'o-', c='#2e7d32', lw=2, label='Suy ra bằng luật: đúng nước')
ax.plot(x, sim['Suy ra: không chắc → chụp lại'] * 100, 's-', c='#f9a825', label='Suy ra: không chắc → chụp lại')
ax.plot(x, sim['Suy ra: SAI mà không biết'] * 100, '^-', c='#c62828', label='Suy ra: sai mà không biết')
ax.axvline(3.9, c='#1565c0', ls=':', lw=1.5); ax.text(4.1, 50, 'mức của YOLO11n\n(Recall 0.961)', color='#1565c0', fontsize=9)
ax.set_xlabel('Tỉ lệ bỏ sót mỗi quân e (%)'); ax.set_ylabel('% số lượt'); ax.set_ylim(-3, 103); ax.grid(alpha=0.3)
ax.legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=2); ax.set_title('Hiệu quả sửa lỗi nhận diện bằng luật cờ (mô phỏng)')
plt.tight_layout(); plt.show()

> **Đọc kết quả** (600 thế cờ, trung bình 21 quân):
> - Ở mức lỗi tương đương mô hình Bước 2 ($e = 4\%$): **tin thẳng ảnh** chỉ cho thế cờ đúng ở **33.5%** số lượt, vì bàn nhiều quân nên thường có ít nhất 1 ô sai. **Suy ra bằng luật** tìm đúng nước ở **88.5%**, **11.5%** còn lại được báo *không chắc* để robot chụp lại, và **0%** sai mà không biết.
> - Lỗi càng nhiều thì tỉ lệ *chụp lại* càng tăng, nhưng tỉ lệ **sai mà không biết** vẫn gần 0 (0.2% ở $e = 12\%$). Đây là tính chất quan trọng nhất: hệ thống thà hỏi lại chứ không đi theo thế cờ sai.
> - Mỗi lần chụp lại có lỗi ngẫu nhiên khác nhau, nên với 11.5% phải chụp lại, chụp thêm 1–2 lần là gần như luôn đủ (xem cột *số lần chụp* ở mục 6.6).
>
> **Giới hạn của mô phỏng:** lỗi thật không độc lập giữa các quân (ánh sáng, che khuất làm sai cả vùng) và ván ngẫu nhiên khác ván người chơi thật. Kết quả cần được đo lại trên ảnh thật (Bước 4 và đánh giá trên robot).

---
# 6. Engine: tìm nước đi tiếp theo

## 6.1 Chọn engine

| Engine | Sức mạnh | Giao thức | Chạy trên Jetson Nano (aarch64) | Ghi chú |
|---|---|---|---|---|
| **Pikafish** | Mạnh nhất hiện nay (mã nguồn mở) | UCI | Build chéo (cross-compile) tĩnh → chép sang (mục 7) | Dựa trên Stockfish, đánh giá bằng mạng NNUE |
| ElephantEye | Khá (đời cũ) | UCCI | Biên dịch được bằng GCC cũ | Phương án dự phòng thứ 2 |
| `brain/simple.py` | Yếu (alpha-beta 3 nước + tìm kiếm tĩnh) | Python + UCI | Chạy ngay, không cần biên dịch | Dự phòng cuối cùng, dùng để thử nghiệm |

Không tự viết engine vì đây không phải mục tiêu của đề tài: đóng góp của đề tài là tầng nhận diện và kiểm tra (mục 1). Engine được tích hợp qua **UCI**, giao thức chuẩn bằng văn bản (`position fen … moves …` → `go depth N` → `bestmove h2e2`), nên có thể thay engine khác mà không phải sửa phần còn lại.

## 6.2 Build Pikafish

Ô dưới chạy `tools/build_pikafish.sh x86`: tải mã nguồn và mạng `pikafish.nnue`, build cho máy đang chạy (≈ 2–3 phút trên Colab). Nếu không có Pikafish, notebook vẫn chạy được bằng engine dự phòng (kết quả mục 6 khi đó chỉ để thử luồng chạy).

In [ ]:
PIKAFISH = f'{REPO}/dist/pikafish_x86/pikafish'
NNUE = f'{REPO}/dist/pikafish_x86/pikafish.nnue'
if not os.path.exists(PIKAFISH) and IN_COLAB:
    t0 = time.time()
    subprocess.run(['sh', 'tools/build_pikafish.sh', 'x86'], check=True, stdout=subprocess.DEVNULL)
    print('Build xong sau %.0f s' % (time.time() - t0))

HAVE_PIKAFISH = os.path.exists(PIKAFISH) and os.path.exists(NNUE)
if HAVE_PIKAFISH:
    eng = UciEngine(PIKAFISH, eval_file=NNUE, threads=2, hash_mb=64, timeout=60)
    print('Engine:', eng.name, '| kích thước mạng NNUE: %.1f MB' % (os.path.getsize(NNUE) / 2**20))
else:
    print('⚠ Không có Pikafish → dùng engine dự phòng brain/simple.py (chỉ để thử luồng chạy)')
    eng = UciEngine(sys.executable, args=['-m', 'brain.simple'], cwd=REPO, timeout=120)
    print('Engine:', eng.name)

## 6.3 Thời gian suy nghĩ theo độ sâu

Độ sâu (*depth*) là số nước engine tính trước. Sâu hơn → mạnh hơn nhưng lâu hơn, thời gian tăng gần theo hàm mũ. Đo trên 3 thế cờ (khai cuộc + 2 thế giữa trận) để chọn độ sâu mặc định cho robot: người chơi không muốn chờ quá vài giây.

In [ ]:
BENCH_FENS = [R.START,
              'r1bakab1r/9/1cn4c1/p1p1p1p1p/9/6P2/P1P1P3P/1C2C1N2/9/RNBAKAB1R b - - 0 1',
              '2bakab2/9/2n1c4/p3p1p1p/2p6/6P2/P1P1P3P/4C1N2/4A4/2BAK1B2 w - - 0 1']
DEPTHS = [4, 6, 8, 10, 12, 14] if HAVE_PIKAFISH else [1, 2]
rows = []
for i, fen in enumerate(BENCH_FENS):
    eng.new_game()
    for d in DEPTHS:
        r = eng.best_move(fen, depth=d)
        b, s = R.parse_fen(fen)
        rows.append({'thế': i, 'độ sâu': d, 'nước': r.move, 'ký hiệu': R.move_to_vn(b, R.uci_to_move(r.move)),
                     'điểm (cp)': r.score_cp, 'số nút': r.nodes, 'thời gian (ms)': round(r.time_ms)})
bench = pd.DataFrame(rows); display(bench)

fig, ax = plt.subplots(figsize=(7, 3.8))
for i, g in bench.groupby('thế'):
    ax.plot(g['độ sâu'], g['thời gian (ms)'], 'o-', label=f'thế {i}')
ax.set_yscale('log'); ax.set_xlabel('độ sâu'); ax.set_ylabel('thời gian (ms, thang log)'); ax.grid(alpha=0.3, which='both')
ax.set_title('Thời gian tìm nước theo độ sâu (' + ('Pikafish, CPU Colab' if HAVE_PIKAFISH else 'engine dự phòng') + ')')
ax.legend(); plt.tight_layout(); plt.show()

> Số liệu trên là CPU của Colab. Jetson Nano (4 nhân Cortex-A57, 1.43 GHz) chậm hơn nhiều lần, và còn phải chia CPU với camera + YOLO. Độ sâu mặc định cho robot được chọn sau khi chạy `tools/bench_engine.py` trên chính robot (mục 7).

## 6.4 Đối chiếu bộ sinh nước đi với Pikafish

Sinh 300 thế cờ giữa trận ngẫu nhiên (có chiếu tướng, tốt qua sông, quân bị cản…), đếm perft độ sâu 2 bằng cả `board/rules.py` và Pikafish. Hai bên được viết độc lập; nếu khớp hết thì gần như chắc chắn luật được cài đúng.

In [ ]:
check_pos = random_positions(300 if HAVE_PIKAFISH else 20, seed=7)
mismatch, in_check = [], 0
t0 = time.time()
for b, s, _ in check_pos:
    fen = R.to_fen(b, s)
    in_check += R.in_check(b, s)
    mine, theirs = R.perft(b, s, 2), eng.perft(fen, 2)
    if mine != theirs:
        mismatch.append((fen, mine, theirs))
print(f'{len(check_pos)} thế cờ ({in_check} thế đang bị chiếu) | lệch: {len(mismatch)} | {time.time() - t0:.0f} s')
for m in mismatch[:5]:
    print('  LỆCH', m)

## 6.5 Ván tự đánh: mọi nước của engine phải hợp lệ theo `board/rules.py`

Cho engine tự đánh với chính nó (độ sâu thấp cho nhanh). Mỗi nước engine trả về đều được kiểm tra lại bằng `R.legal_moves()` — đây là phép thử tích hợp giữa hai tầng. Kết quả ghi bằng ký hiệu tiếng Việt.

In [ ]:
SELF_DEPTH, MAX_PLY = (6, 80) if HAVE_PIKAFISH else (1, 12)
b, s, log, hist = R.start_board(), R.RED, [], []
eng.new_game()
for ply in range(MAX_PLY):
    st = R.game_status(b, s)
    if st != 'ongoing':
        print('Kết thúc:', st, '— bên thua:', 'Đỏ' if s == R.RED else 'Đen'); break
    r = eng.best_move(R.START, moves=hist, depth=SELF_DEPTH)
    mv = R.uci_to_move(r.move)
    assert mv in R.legal_moves(b, s), f'Engine đi nước không hợp lệ: {r.move}'
    log.append(R.move_to_vn(b, mv)); hist.append(r.move)
    b, s = R.make_move(b, mv), R.other(s)
print(f'{len(hist)} nước, tất cả hợp lệ ✅')
pairs = [f'{i + 1:>2}. {log[2 * i]:<22} {log[2 * i + 1] if 2 * i + 1 < len(log) else ""}' for i in range((len(log) + 1) // 2)]
print('\n'.join(pairs[:20]) + ('\n...' if len(pairs) > 20 else ''))
fig, ax = plt.subplots(figsize=(4, 4.6)); draw_board(ax, b, f'Thế cờ sau {len(hist)} nước'); plt.show()

## 6.6 Ghép lại: vòng lặp người – robot (`GameTracker`)

Đây là giao diện mà chương trình chính của robot sẽ gọi. Người cầm quân Đỏ, robot cầm quân Đen:

```python
game = GameTracker(robot_side='b')
r = game.observe(board_from_camera)    # người vừa đi: suy ra nước, sửa lỗi nhận diện
if r['status'] in ('moved', 'corrected'):
    res = game.suggest(engine, depth=8) # nước tiếp theo cho robot
    game.apply(res.move)                # sau khi tay máy đi xong
elif r['status'] == 'uncertain':
    ...                                 # chụp lại
```

Ô dưới mô phỏng 12 lượt: "người" đi ngẫu nhiên, ảnh bị nhiễu ở mức $e = 4\%$ như mục 5.3, gặp `uncertain` thì "chụp lại" (sinh ảnh nhiễu mới).

In [ ]:
game = GameTracker(robot_side=R.BLACK)
rnd, rows = random.Random(3), []
eng.new_game()
for turn in range(12):
    if R.game_status(game.board, game.side) != 'ongoing':
        break
    human_mv = rnd.choice(R.legal_moves(game.board, game.side))       # "người" Đỏ đi
    truth = R.make_move(game.board, human_mv)
    for shot in range(1, 6):                                          # chụp tối đa 5 lần
        obs = corrupt(truth, 0.04, rnd)
        r = game.observe(obs)
        if r['status'] in ('moved', 'corrected', 'game_over'):
            break
    if r['status'] == 'game_over':
        break
    assert r['move'] == human_mv
    if R.game_status(game.board, game.side) != 'ongoing':
        rows.append({'lượt': turn + 1, 'người (Đỏ)': r['vn'], 'nhận diện': r['status'], 'số lần chụp': shot}); break
    res = game.suggest(eng, depth=8 if HAVE_PIKAFISH else 1)
    robot_vn = game.apply(res.move)
    rows.append({'lượt': turn + 1, 'người (Đỏ)': r['vn'], 'nhận diện': r['status'], 'ô đã sửa': r['cost'],
                 'số lần chụp': shot, 'robot (Đen)': robot_vn, 'nước UCI': res.move, 'điểm robot (cp)': res.score_cp})
display(pd.DataFrame(rows))
print('FEN hiện tại:', game.fen)

---
# 7. Đưa engine lên Jetson Nano

Robot **không có mạng** và chạy JetPack 4.6 (Ubuntu 18.04, GCC 7). Pikafish cần C++17 mới hơn, nên **không build trên robot**. Thay vào đó:

1. Build chéo (*cross-compile*) cho `aarch64` trên Colab / laptop bằng `aarch64-linux-gnu-g++`, **link tĩnh** (`-static`) → file chạy không phụ thuộc thư viện của Ubuntu 18.04.
2. `ARCH=armv8`: Cortex-A57 của Jetson Nano có NEON nhưng không có lệnh *dotprod*.
3. `-DNNUE_EMBEDDING_OFF`: không nhúng mạng vào file chạy; mạng `pikafish.nnue` để cạnh file chạy và nạp lúc khởi động. ⚠ Mạng phải tải **cùng lúc** với mã nguồn (định dạng mạng thay đổi theo phiên bản).
4. Chép `pikafish_jetson.tar.gz` sang robot qua NoMachine, giải nén vào `~/pikafish/`.

Ô dưới build và tải gói về máy (≈ 3–5 phút).

In [ ]:
BUILD_JETSON = IN_COLAB      # đặt True trên laptop Ubuntu nếu có quyền cài g++-aarch64-linux-gnu
if BUILD_JETSON:
    subprocess.run(['sh', 'tools/build_pikafish.sh', 'jetson'], check=True)
    print(open('dist/pikafish_jetson/VERSION.txt').read())
    print(subprocess.run(['file', 'dist/pikafish_jetson/pikafish'], capture_output=True, text=True).stdout)
    if IN_COLAB:
        from google.colab import files
        files.download('dist/pikafish_jetson.tar.gz')

### Trên robot

```bash
tar xzf pikafish_jetson.tar.gz -C ~/ && mv ~/pikafish_jetson ~/pikafish
~/pikafish/pikafish <<< "uci"                     # phải in ra "uciok"
cd ~/xiangqi-sorting-jetmax
python3 tools/bench_engine.py --engine ~/pikafish/pikafish --depths 6 8 10 12
python3 tools/bench_engine.py --simple            # so sánh với engine dự phòng
```

Kết quả ghi vào `runs/bench_engine.csv`. Điền vào bảng dưới để chọn độ sâu mặc định (mục tiêu: ≤ 3 giây mỗi nước, đủ RAM khi chạy cùng YOLO):

| Độ sâu | Thời gian TB (ms) | RAM engine (MB) | Ghi chú |
|---|---|---|---|
| 6 | | | |
| 8 | | | |
| 10 | | | |
| 12 | | | |

In [ ]:
eng.close()

---
# Tổng kết

| Hạng mục | Kết quả |
|---|---|
| Bộ sinh nước đi | Perft thế khai cuộc khớp chuẩn ở độ sâu 1–4 (44 / 1 920 / 79 666 / 3 290 240) |
| Đối chiếu Pikafish | Perft độ sâu 2 khớp trên các thế cờ giữa trận ngẫu nhiên (mục 6.4) |
| Kiểm tra thế cờ | 5 nhóm luật: số lượng quân, tướng, vị trí sĩ/tượng/tốt, 2 tướng đối mặt, bên vừa đi bị chiếu |
| Sửa lỗi nhận diện (mô phỏng, $e = 4\%$) | Tin thẳng ảnh đúng **33.5%** → suy ra bằng luật đúng **88.5%**, chụp lại 11.5%, sai mà không biết **0%** (mục 5.3) |
| Engine | Pikafish qua UCI; dự phòng `brain/simple.py`; gói build sẵn cho Jetson Nano (aarch64, link tĩnh) |
| Giao diện cho robot | `GameTracker.observe()` → `suggest()` → `apply()` |

**Hạn chế và hướng phát triển:**
1. Chưa xử lý luật lặp nước / đuổi bắt dai (Pikafish tự xử lý khi nhận đủ chuỗi nước đi, nhưng `board/rules.py` thì chưa).
2. Khoảng cách Hamming coi mọi ô như nhau; có thể dùng confidence của YOLO làm trọng số (ô có quân confidence thấp thì "rẻ" khi sửa).
3. Kết quả mục 5.3 là mô phỏng; cần đo lại bằng ảnh thật từ camera robot.
4. Đo tốc độ Pikafish trên Jetson Nano và chọn độ sâu mặc định (mục 7).

---
# Tài liệu tham khảo (Sources)

1. Pikafish — engine cờ tướng mã nguồn mở: https://github.com/official-pikafish/Pikafish ; tài liệu: https://github.com/official-pikafish/Pikafish/wiki
2. Mạng NNUE của Pikafish: https://github.com/official-pikafish/Networks
3. Chessprogramming wiki — *Chinese Chess Perft Results*: https://chessprogramming.org/Chinese_Chess_Perft_Results
4. Chessprogramming wiki — *Perft*: https://www.chessprogramming.org/Perft_Results
5. ElephantEye (engine dự phòng, giao thức UCCI): https://github.com/xqbase/eleeye
6. H. G. Muller — *UCCI protocol / XQ engines*: https://home.hccnet.nl/h.g.muller/XQucci.html
7. Nasu Y. — *Efficiently Updatable Neural-Network-based Evaluation Functions for Computer Shogi* (NNUE), 2018.
8. Knuth D. E., Moore R. W. — *An Analysis of Alpha-Beta Pruning*, Artificial Intelligence 6(4), 1975.